In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from model import NSConfig, NsModel, CUR_DIR

In [2]:
from tokenizers import Tokenizer
tokenizer = Tokenizer.from_file(f"{CUR_DIR}/model/tokenizer_fixed.json")

In [3]:
cfg = NSConfig()

device = "cuda:1"

model = NsModel(cfg)
model.to(device)

model.load_state_dict(
    torch.load(f"{CUR_DIR}/model/pretrain_ns_model_epoch_1.bin", map_location=device, weights_only=True)
)
model.to(device)
model.eval()


NsModel(
  (emb): Embedding(32000, 768)
  (layers): ModuleList(
    (0-11): 12 x TransformerBlock(
      (rms_norm1): RMSNorm()
      (attn): GQAAttention(
        (w_q): Linear(in_features=768, out_features=768, bias=False)
        (w_k): Linear(in_features=768, out_features=128, bias=False)
        (w_v): Linear(in_features=768, out_features=128, bias=False)
        (w_o): Linear(in_features=768, out_features=768, bias=False)
      )
      (rms_norm2): RMSNorm()
      (ffn): FeedForward(
        (fc1): Linear(in_features=768, out_features=2048, bias=False)
        (fc2): Linear(in_features=768, out_features=2048, bias=False)
        (fc3): Linear(in_features=2048, out_features=768, bias=False)
      )
    )
  )
  (final_norm): RMSNorm()
  (lm_out): Linear(in_features=768, out_features=32000, bias=False)
)

In [4]:
prompt = "The capital of France is"

input_ids = tokenizer.encode(prompt).ids

eos_token_id = tokenizer.token_to_id("<eos>")
input_ids = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(device)

In [5]:
# no kv-cache

max_output = 200

with torch.no_grad():

    
    for _ in range(max_output):

        logits = model.forward(input_ids)

        logits = logits[:, -1, :]

        # 贪心
        # next_token = torch.argmax(logits, dim=-1, keepdim=True)

        # 概率
        probs = torch.softmax(
            logits,
            dim=-1
        )
        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        if next_token.item() == eos_token_id:
            print("-------------end-----------------------")
            break
        
        input_ids = torch.cat([input_ids, next_token], dim=1)




output = tokenizer.decode(input_ids.squeeze(0).cpu().numpy())
print(output)


-------------end-----------------------
 The capital of France is Temple of Pondus and Dominos. Blonde Laura has two lovely eyes and 12 eyes and Kate is eighteen too! Carrie would recommend this sparkling collection too.


In [6]:
# enable kvcache

max_output = 1500

with torch.no_grad():


    # 1. prefill
    logits, past_key_values = model.forward(input_ids, use_cache=True)

    generated = input_ids


    for _ in range(max_output):

        logits = logits[:, -1, :]


        # 概率
        probs = torch.softmax(
            logits,
            dim=-1
        )
        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        generated = torch.cat([generated, next_token], dim=1)

        if next_token.item() == eos_token_id:
            print("-------------end-----------------------")
            break

        logits, past_key_values = model(next_token, past_key_values=past_key_values, use_cache=True)

    
output = tokenizer.decode(generated.squeeze(0).cpu().numpy())
print(output)


 the cat jump over the dog field.
Biopa wondered how she was remaining over the rock she started out fighting every week. I tried to contain my anger, I participated in as slams as I had this deluded me into using proxies if I needed to. Sometimes I have a tendency to run my adkins to me, I confess that when it comes to disciplinary calls in Australian dialect, both shy and incomplete. So tired of the oh so far more I started with remarks which I knw ing in between. Maybe when we took the boat, we stopped the singing.
That delighted the ears.
Instead I let them join us.
So decided to reminisce that the sea had a rather militant spirits.
Thanks to you for clearing any platitudes that a certain crook reacted by
as mere guides and words, perhaps I should bookmarkLast Blooming Agony and reply to T.R.Enflavin's story highlighting the amazing Kahalleys!
________________Bent on speaking in the newest Bubby sequel video I wrote at length with Waddy Eisenhower.
Since then, it's not always easy 

In [6]:
# inference with attention mask

prompts = [
    "the cat jump over the dog",
    "my name is sam, and i watch the movie on tv"
]

pad_token_id = tokenizer.token_to_id("<pad>")

input_ids = tokenizer.encode_batch(prompts, add_special_tokens=False)

max_len = max(len(ids) for ids in input_ids)

input_list = []
mask_list = []

for item in input_ids:
    ids = item.ids
    pad_len = max_len - len(ids)

    if pad_len > 0:
        pad_ids = torch.tensor(ids + [pad_token_id] * pad_len, dtype=torch.long)
        attn_mask = torch.tensor([1] * len(ids) + [0] * pad_len, dtype=torch.long)
        
    else:
        pad_ids = torch.tensor(ids , dtype=torch.long)
        attn_mask = torch.tensor([1] * max_len, dtype=torch.long)

    input_list.append(pad_ids)
    mask_list.append(attn_mask)

input_tensor = torch.vstack(input_list).to(device)
attn_mask_tensor = torch.vstack(mask_list).to(device)


In [32]:
input_tensor 

tensor([[  266,  3200,  3978,   629,   266,  3050,     1,     1,     1,     1,
             1,     1],
        [  518,  1562,   325,  6143,    15,   291,  1172,  2176,   266,  2966,
           324, 20199]], device='cuda:1')

In [33]:
attn_mask_tensor

tensor([[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:1')

In [7]:
# no kv-cache + atten_mask

max_output = 200

with torch.no_grad():

    
    for _ in range(max_output):

        logits = model.forward(input_tensor, attention_mask=attn_mask_tensor)
        # logits = model.forward(input_tensor, attention_mask=None)

        logits = logits[:, -1, :]

        # 概率采样
        probs = torch.softmax(
            logits,
            dim=-1
        )
        next_token = torch.multinomial(
            probs,
            num_samples=1
        )
        
        input_tensor = torch.cat([input_tensor, next_token], dim=1)
        attn_mask_tensor = torch.cat([attn_mask_tensor, torch.ones((attn_mask_tensor.shape[0], 1), device=attn_mask_tensor.device, dtype=attn_mask_tensor.dtype)], dim=1)

outputs = input_tensor.cpu().numpy()

for output in outputs:
    print(tokenizer.decode(output))
    print("----------------------------")

 the cat jump over the dog then tell her those were her cat loves. Inside today’s effort to get a pet to a decision is blind obedience. The hard part is just getting the food on the table. The cat is the dog. It has to be willing. There are important ethical issues between the animals that has to be answered. Are we living a blanket comfort zone and limiting or are we dead simple? Are we safe, get along, keep our favorite bump on the table, and talk big about diapers? And we both have health issues. These issues are privately and fairly universal. The truth is the cats are happier, contain more or less negative feelings. With the increased pressures on the cats, coughing up congestion, and low insomnia, our stores have gotten fuller with fewer or no visits. Every new item this spring is an improvement. A constant eye, a soft spot for all our troubles, goes dry as insomnia damages the pet. When it comes can a responsibility grow. It
----------------------------
 my name is sam, and i wa

In [8]:

prompts = [
    "the cat jump over the dog",
    "my name is sam, and i watch the movie on tv"
]

pad_token_id = tokenizer.token_to_id("<pad>")

input_ids = tokenizer.encode_batch(prompts, add_special_tokens=False)

max_len = max(len(ids) for ids in input_ids)

input_list = []
mask_list = []

for item in input_ids:
    ids = item.ids
    pad_len = max_len - len(ids)

    if pad_len > 0:
        pad_ids = torch.tensor(ids + [pad_token_id] * pad_len, dtype=torch.long)
        attn_mask = torch.tensor([1] * len(ids) + [0] * pad_len, dtype=torch.long)
        
    else:
        pad_ids = torch.tensor(ids , dtype=torch.long)
        attn_mask = torch.tensor([1] * max_len, dtype=torch.long)

    input_list.append(pad_ids)
    mask_list.append(attn_mask)

input_tensor = torch.vstack(input_list).to(device)
attn_mask_tensor = torch.vstack(mask_list).to(device)

In [43]:
input_tensor

tensor([[  266,  3200,  3978,   629,   266,  3050,     1,     1,     1,     1,
             1,     1,   354],
        [  518,  1562,   325,  6143,    15,   291,  1172,  2176,   266,  2966,
           324, 20199,   291]], device='cuda:1')

In [ ]:
# kv-cache + atten_mask

max_output = 200


with torch.no_grad():


    # 1. prefill
    logits, past_key_values = model.forward(input_tensor, use_cache=True, attention_mask=attn_mask_tensor)

    generated = input_ids


    for _ in range(max_output):

        logits = logits[:, -1, :]


        # 概率
        probs = torch.softmax(
            logits,
            dim=-1
        )
        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        input_tensor = torch.cat([input_tensor, next_token], dim=1)
        attn_mask_tensor = torch.cat([attn_mask_tensor, torch.ones((attn_mask_tensor.shape[0], 1), device=attn_mask_tensor.device, dtype=attn_mask_tensor.dtype)], dim=1)
        # NOTE: 使用kvcache时，只传入new token
        logits, past_key_values = model(next_token, past_key_values=past_key_values, use_cache=True, attention_mask=attn_mask_tensor)

outputs = input_tensor.cpu().numpy()

for output in outputs:
    print(tokenizer.decode(output))
    print("----------------------------")

 the cat jump over the dog for me, period. I love the last stout's, but they are too narrow, too loose, and lacking the playability of tweed and I don't ever get to either move -- they also raise the hips blades much less, just......BUT .... was also not recommended to move very far with the offer of a taller dog (which was a bit less than he thought) so I know tendancy to clench nubs and pound things until he grows up in the shadow.
So despite being true to Celestial Racism and Pop Up Adult Love, I'm glad that you like your dogs and not 720's. Give your house some heartburn over a toddler co-sleeper, and a new addition to the bench.
Today on their new sites their little ones look like they got lazy. Did they get all that business?
Geix is riding the rides with their new dolt Nars, and I also let him out of CA with maybe his
----------------------------
 my name is sam, and i watch the movie on tv. This movie walks you through how i watch the movie and how i watch the part of a brat. I

: 